# Landslide combined-class step 07: baseline EAD minimum/maximum comparison maps

Compares baseline EAD outputs from the combined-class minimum and maximum scenarios.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, TwoSlopeNorm, LinearSegmentedColormap
from matplotlib.cm import ScalarMappable
from IPython.display import display


In [ ]:
base_path = Path('/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers')
network_csv = base_path / 'dphil_common_cross_cutting/common_incoming_data/networks/network_layers_hazard_intersections_details.csv'
jamaica_boundary_path = base_path / 'dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg'
common_data_root = base_path / 'dphil_common_cross_cutting/common_incoming_data'

minimum_asset_ead_csv = base_path / 'dphil_paper_3/results/02_damage_estimates/landslide_damages/results_landslide_minimum_scenario_combined_class/damage_estimates/landslide_ead_asset_level_usd_combined_class.csv'
maximum_asset_ead_csv = base_path / 'dphil_paper_3/results/02_damage_estimates/landslide_damages/results_landslide_maximum_scenario_combined_class/damage_estimates/landslide_ead_asset_level_usd_combined_class.csv'

comparison_output_dir = base_path / 'dphil_paper_3/results/02_damage_estimates/landslide_damages/min_max_comparison_maps_source_and_runout_zones_combined_class'
comparison_output_dir.mkdir(parents=True, exist_ok=True)

for required_file in [network_csv, jamaica_boundary_path, minimum_asset_ead_csv, maximum_asset_ead_csv]:
    if not required_file.exists():
        raise FileNotFoundError(f'Missing required file: {required_file}')

print('Minimum scenario asset EAD:', minimum_asset_ead_csv)
print('Maximum scenario asset EAD:', maximum_asset_ead_csv)
print('Output directory:', comparison_output_dir)

# Performance controls
save_map_layers = False  # set True if full asset map layers are needed; files can be very large
max_plot_features = 250000  # random sample cap for plotting speed


In [ ]:
network_details = pd.read_csv(network_csv)[[
    'sector', 'asset_description', 'asset_gpkg', 'asset_layer', 'asset_id_column', 'path'
]].drop_duplicates().copy()


def resolve_network_asset_file(asset_relative_path):
    relative_asset_path = Path(asset_relative_path)
    asset_file_in_common_data = common_data_root / relative_asset_path
    asset_file_in_nested_networks_folder = common_data_root / 'networks' / relative_asset_path

    if asset_file_in_common_data.exists():
        return asset_file_in_common_data
    if asset_file_in_nested_networks_folder.exists():
        return asset_file_in_nested_networks_folder

    raise FileNotFoundError(
        f"Could not find asset file '{relative_asset_path}'. Checked: {asset_file_in_common_data} ; {asset_file_in_nested_networks_folder}"
    )


def build_baseline_map_layer(asset_ead_csv: Path, scenario_label: str) -> gpd.GeoDataFrame:
    asset_ead = pd.read_csv(asset_ead_csv, low_memory=False)
    required_columns = ['Asset', 'Layer', 'Asset_ID', 'EAD_Baseline_USD']
    missing_columns = [required_column for required_column in required_columns if required_column not in asset_ead.columns]
    if missing_columns:
        raise KeyError(f'{asset_ead_csv.name} missing columns: {missing_columns}')

    map_layers = []

    for network_layer in network_details.itertuples(index=False):
        ead_subset = asset_ead.loc[
            (asset_ead['Asset'] == network_layer.asset_gpkg) & (asset_ead['Layer'] == network_layer.asset_layer),
            ['Asset_ID', 'EAD_Baseline_USD']
        ].copy()
        if ead_subset.empty:
            continue

        asset_file = resolve_network_asset_file(network_layer.path)
        asset_geodataframe = gpd.read_file(asset_file, layer=network_layer.asset_layer)
        if network_layer.asset_id_column not in asset_geodataframe.columns:
            continue

        asset_geodataframe = asset_geodataframe[[network_layer.asset_id_column, 'geometry']].copy()
        asset_geodataframe = gpd.GeoDataFrame(asset_geodataframe, geometry='geometry', crs=asset_geodataframe.crs)
        if asset_geodataframe.crs is not None:
            asset_geodataframe = asset_geodataframe.to_crs('EPSG:3448')
        else:
            asset_geodataframe = asset_geodataframe.set_crs('EPSG:3448', allow_override=True)

        asset_geodataframe['asset_join_key'] = asset_geodataframe[network_layer.asset_id_column].astype(str)
        ead_subset['asset_join_key'] = ead_subset['Asset_ID'].astype(str)

        joined_assets = asset_geodataframe.merge(ead_subset, on='asset_join_key', how='inner')
        if joined_assets.empty:
            continue

        joined_assets['Scenario'] = scenario_label
        joined_assets['Sector'] = network_layer.sector
        joined_assets['Subsector'] = network_layer.asset_description
        joined_assets['Asset'] = network_layer.asset_gpkg
        joined_assets['Layer'] = network_layer.asset_layer

        map_layers.append(joined_assets[[
            'Scenario', 'Sector', 'Subsector', 'Asset', 'Layer',
            network_layer.asset_id_column, 'Asset_ID', 'EAD_Baseline_USD', 'geometry'
        ]].rename(columns={network_layer.asset_id_column: 'Asset_ID_Source'}))

    if not map_layers:
        raise ValueError(f'No map layers built for {scenario_label}')

    return gpd.GeoDataFrame(pd.concat(map_layers, ignore_index=True), geometry='geometry', crs='EPSG:3448')


In [ ]:
baseline_minimum_layer = build_baseline_map_layer(minimum_asset_ead_csv, 'minimum')
baseline_maximum_layer = build_baseline_map_layer(maximum_asset_ead_csv, 'maximum')

print('Minimum map features:', len(baseline_minimum_layer))
print('Maximum map features:', len(baseline_maximum_layer))

display(baseline_minimum_layer[['EAD_Baseline_USD']].describe(percentiles=[0.5, 0.9, 0.99]))


In [ ]:
# Optional: save full map layers for reuse (can be large, especially with building polygons)
if save_map_layers:
    minimum_layer_output = comparison_output_dir / 'baseline_ead_minimum_asset_map_layers_combined_class.gpkg'
    maximum_layer_output = comparison_output_dir / 'baseline_ead_maximum_asset_map_layers_combined_class.gpkg'

    baseline_minimum_layer.to_file(minimum_layer_output, driver='GPKG')
    baseline_maximum_layer.to_file(maximum_layer_output, driver='GPKG')

    print('Saved:', minimum_layer_output)
    print('Saved:', maximum_layer_output)
else:
    print('Skipped saving full GPKG map layers (save_map_layers=False).')


In [ ]:
jamaica_boundary = gpd.read_file(jamaica_boundary_path).to_crs('EPSG:3448')

all_baseline_values = pd.concat([
    baseline_minimum_layer['EAD_Baseline_USD'].astype(float),
    baseline_maximum_layer['EAD_Baseline_USD'].astype(float)
], ignore_index=True).fillna(0.0)

# Robust shared color scale for clear comparison.
display_quantile = 0.995
maximum_display_usd = float(all_baseline_values.quantile(display_quantile))
if maximum_display_usd <= 0:
    maximum_display_usd = float(all_baseline_values.max()) if float(all_baseline_values.max()) > 0 else 1.0

if maximum_display_usd >= 1e6:
    unit_factor = 1e6
    unit_label = 'USD millions'
elif maximum_display_usd >= 1e3:
    unit_factor = 1e3
    unit_label = 'USD thousands'
else:
    unit_factor = 1.0
    unit_label = 'USD'

maximum_display_value = maximum_display_usd / unit_factor
baseline_norm = Normalize(vmin=0, vmax=maximum_display_value)
baseline_colormap = plt.cm.viridis


def sample_for_plot(asset_layer, max_features):
    if len(asset_layer) <= max_features:
        return asset_layer
    return asset_layer.sample(n=max_features, random_state=42)


def add_north_arrow(axis):
    axis.annotate(
        'N',
        xy=(0.94, 0.90),
        xytext=(0.94, 0.78),
        xycoords='axes fraction',
        textcoords='axes fraction',
        arrowprops=dict(facecolor='black', edgecolor='black', width=3, headwidth=10, headlength=12),
        ha='center',
        va='center',
        fontsize=12,
        fontweight='bold',
        zorder=20,
    )


def add_scale_bar(axis, length_km=25):
    x_minimum, x_maximum = axis.get_xlim()
    y_minimum, y_maximum = axis.get_ylim()
    x_range = x_maximum - x_minimum
    y_range = y_maximum - y_minimum
    length_metres = length_km * 1000.0
    x_start = x_minimum + 0.08 * x_range
    y_start = y_minimum + 0.07 * y_range
    tick_height = 0.01 * y_range

    axis.plot([x_start, x_start + length_metres], [y_start, y_start], color='black', linewidth=3, solid_capstyle='butt', zorder=20)
    axis.plot([x_start, x_start], [y_start - tick_height, y_start + tick_height], color='black', linewidth=2, zorder=20)
    axis.plot([x_start + length_metres, x_start + length_metres], [y_start - tick_height, y_start + tick_height], color='black', linewidth=2, zorder=20)
    axis.text(
        x_start + (length_metres / 2.0),
        y_start + (2.2 * tick_height),
        f'{length_km:g} km',
        ha='center',
        va='bottom',
        fontsize=9,
        bbox=dict(facecolor='white', edgecolor='none', alpha=0.75, pad=1.5),
        zorder=20,
    )


def apply_jamaica_map_formatting(axis):
    add_north_arrow(axis)
    add_scale_bar(axis)
    axis.set_axis_off()


def plot_baseline_panel(axis, asset_layer, title):
    plot_layer = sample_for_plot(asset_layer.copy(), max_plot_features)
    if len(plot_layer) < len(asset_layer):
        print(f"{title}: plotting sample {len(plot_layer):,} / {len(asset_layer):,} features")

    plot_layer['plot_value'] = (plot_layer['EAD_Baseline_USD'] / unit_factor).clip(0, maximum_display_value)

    axis.set_facecolor('#ffffff')
    jamaica_boundary.boundary.plot(ax=axis, color='#bdbdbd', linewidth=0.45, zorder=1)

    geometry_type = plot_layer.geometry.geom_type.astype(str)
    polygons = plot_layer[geometry_type.str.contains('Polygon', na=False)]
    lines = plot_layer[geometry_type.str.contains('LineString', na=False)]
    points = plot_layer[geometry_type.str.contains('Point', na=False)]

    if not polygons.empty:
        polygons.plot(ax=axis, column='plot_value', cmap=baseline_colormap, norm=baseline_norm, linewidth=0.10, edgecolor='none', alpha=0.9, zorder=2)
    if not lines.empty:
        lines.plot(ax=axis, column='plot_value', cmap=baseline_colormap, norm=baseline_norm, linewidth=0.85, alpha=0.95, zorder=3)
    if not points.empty:
        points.plot(ax=axis, column='plot_value', cmap=baseline_colormap, norm=baseline_norm, markersize=14, alpha=0.95, zorder=4)

    axis.set_title(title, fontsize=12)
    apply_jamaica_map_formatting(axis)


figure, axes = plt.subplots(1, 2, figsize=(16, 8), constrained_layout=True)
plot_baseline_panel(axes[0], baseline_minimum_layer, 'Combined-class baseline EAD - Minimum scenario')
plot_baseline_panel(axes[1], baseline_maximum_layer, 'Combined-class baseline EAD - Maximum scenario')

scalar_mappable = ScalarMappable(norm=baseline_norm, cmap=baseline_colormap)
scalar_mappable.set_array([])
colorbar = figure.colorbar(scalar_mappable, ax=axes.ravel().tolist(), fraction=0.03, pad=0.02)
colorbar.set_label(f'Baseline EAD ({unit_label}), clipped at q={display_quantile:.3f}')

side_by_side_png = comparison_output_dir / 'baseline_ead_min_max_side_by_side_shared_scale_combined_class.png'
figure.savefig(side_by_side_png, dpi=300, bbox_inches='tight')
print('Saved:', side_by_side_png)
plt.show()


In [ ]:
# Difference map (maximum - minimum) at asset level.
# Positive values mean the maximum scenario gives higher baseline EAD than the minimum scenario.
minimum_comparison_columns = baseline_minimum_layer[['Asset', 'Layer', 'Asset_ID', 'EAD_Baseline_USD', 'geometry']].copy()
maximum_comparison_columns = baseline_maximum_layer[['Asset', 'Layer', 'Asset_ID', 'EAD_Baseline_USD']].copy()

minimum_comparison_columns['asset_join_key'] = (
    minimum_comparison_columns['Asset'].astype(str)
    + '|'
    + minimum_comparison_columns['Layer'].astype(str)
    + '|'
    + minimum_comparison_columns['Asset_ID'].astype(str)
)
maximum_comparison_columns['asset_join_key'] = (
    maximum_comparison_columns['Asset'].astype(str)
    + '|'
    + maximum_comparison_columns['Layer'].astype(str)
    + '|'
    + maximum_comparison_columns['Asset_ID'].astype(str)
)

baseline_difference = minimum_comparison_columns.merge(
    maximum_comparison_columns[['asset_join_key', 'EAD_Baseline_USD']],
    on='asset_join_key',
    how='inner',
    suffixes=('_minimum', '_maximum')
)

baseline_difference['Baseline_EAD_Difference_Maximum_Minus_Minimum_USD'] = (
    baseline_difference['EAD_Baseline_USD_maximum'] - baseline_difference['EAD_Baseline_USD_minimum']
)

difference_layer = gpd.GeoDataFrame(baseline_difference, geometry='geometry', crs='EPSG:3448')

difference_values = difference_layer['Baseline_EAD_Difference_Maximum_Minus_Minimum_USD'].fillna(0.0)
difference_cap_usd = float(difference_values.abs().quantile(0.995))
if difference_cap_usd <= 0:
    difference_cap_usd = float(difference_values.abs().max()) if float(difference_values.abs().max()) > 0 else 1.0

if difference_cap_usd >= 1e6:
    difference_unit_factor = 1e6
    difference_unit_label = 'USD millions'
elif difference_cap_usd >= 1e3:
    difference_unit_factor = 1e3
    difference_unit_label = 'USD thousands'
else:
    difference_unit_factor = 1.0
    difference_unit_label = 'USD'

difference_display_cap = difference_cap_usd / difference_unit_factor
difference_layer['plot_value'] = (difference_values / difference_unit_factor).clip(-difference_display_cap, difference_display_cap)

difference_plot_layer = sample_for_plot(difference_layer, max_plot_features)
if len(difference_plot_layer) < len(difference_layer):
    print(f"Difference map: plotting sample {len(difference_plot_layer):,} / {len(difference_layer):,} features")

difference_norm = TwoSlopeNorm(vmin=-difference_display_cap, vcenter=0.0, vmax=difference_display_cap)
difference_colormap = LinearSegmentedColormap.from_list('red_white_blue', ['#c81e1e', '#ffffff', '#1f78b4'], N=256)

figure, axis = plt.subplots(figsize=(11, 9))
axis.set_facecolor('#ffffff')
jamaica_boundary.boundary.plot(ax=axis, color='#bdbdbd', linewidth=0.45, zorder=1)

geometry_type = difference_plot_layer.geometry.geom_type.astype(str)
polygons = difference_plot_layer[geometry_type.str.contains('Polygon', na=False)]
lines = difference_plot_layer[geometry_type.str.contains('LineString', na=False)]
points = difference_plot_layer[geometry_type.str.contains('Point', na=False)]

if not polygons.empty:
    polygons.plot(ax=axis, column='plot_value', cmap=difference_colormap, norm=difference_norm, linewidth=0.10, edgecolor='none', alpha=0.9, zorder=2)
if not lines.empty:
    lines.plot(ax=axis, column='plot_value', cmap=difference_colormap, norm=difference_norm, linewidth=0.85, alpha=0.95, zorder=3)
if not points.empty:
    points.plot(ax=axis, column='plot_value', cmap=difference_colormap, norm=difference_norm, markersize=14, alpha=0.95, zorder=4)

scalar_mappable = ScalarMappable(norm=difference_norm, cmap=difference_colormap)
scalar_mappable.set_array([])
colorbar = figure.colorbar(scalar_mappable, ax=axis, fraction=0.03, pad=0.02)
colorbar.set_label(f'Baseline EAD difference (Maximum - Minimum) ({difference_unit_label}), clipped at q=0.995')

axis.set_title('Combined-class baseline EAD difference: Maximum minus Minimum scenario', fontsize=12)
apply_jamaica_map_formatting(axis)

difference_png = comparison_output_dir / 'baseline_ead_difference_maximum_minus_minimum_combined_class.png'
figure.savefig(difference_png, dpi=300, bbox_inches='tight')
print('Saved:', difference_png)
plt.show()


In [ ]:
# Quick comparison table (all-sector totals)
def format_usd_readable(value):
    absolute_value = abs(float(value))
    sign = '-' if float(value) < 0 else ''
    if absolute_value >= 1_000_000_000:
        return f"{sign}${absolute_value / 1_000_000_000:,.2f} billion"
    if absolute_value >= 1_000_000:
        return f"{sign}${absolute_value / 1_000_000:,.2f} million"
    if absolute_value >= 1_000:
        return f"{sign}${absolute_value / 1_000:,.2f} thousand"
    return f"{sign}${absolute_value:,.2f}"


minimum_total = float(baseline_minimum_layer['EAD_Baseline_USD'].sum())
maximum_total = float(baseline_maximum_layer['EAD_Baseline_USD'].sum())
difference_total = maximum_total - minimum_total
percent_difference = (100.0 * difference_total / minimum_total) if minimum_total > 0 else np.nan

comparison_table = pd.DataFrame([
    {'Metric': 'Baseline EAD total (minimum)', 'USD': minimum_total},
    {'Metric': 'Baseline EAD total (maximum)', 'USD': maximum_total},
    {'Metric': 'Difference (maximum - minimum)', 'USD': difference_total},
])
comparison_table['USD_Readable'] = comparison_table['USD'].apply(format_usd_readable)

percent_difference_table = pd.DataFrame([{
    'Metric': 'Percent difference (maximum vs minimum)',
    'Percent': percent_difference,
    'Percent_Label': 'NA' if pd.isna(percent_difference) else f"{percent_difference:.2f}%"
}])

comparison_csv = comparison_output_dir / 'baseline_ead_min_max_total_comparison_combined_class.csv'
comparison_table.to_csv(comparison_csv, index=False)

print('Baseline total comparison:')
display(comparison_table)
display(percent_difference_table)
print('Saved:', comparison_csv)
